In [ ]:
import numpy as np
from gfsupg.solver import CartesianGeometry, FiniteElement1D, Scipy2DFEM
from gfsupg.solver import DeC, DeCSpaceTimeSUPGSolver
# from gfsupg.solver import Numba2DFEM
from gfsupg.problem import LinearAcoustic2D
from gfsupg.plotting import *

from matplotlib import cm
from matplotlib.ticker import LinearLocator

import matplotlib.pyplot as plt
import pickle
import os

In [ ]:

#problem.T_fin = 1.
order=4

FEM1Dx = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
FEM1Dy = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
dec = DeC((order+1)//2,order,"gaussLobatto")


In [ ]:
problem = LinearAcoustic2D("moving_source")
# problem = LinearAcoustic2D("coriolis_vortex_long")

GF = True
stab = "SUPG"#"SUPG"#"OSS"#"OSS"
if GF:
    method_name = stab+"_GF"
else:
    method_name = stab

In [ ]:

Ns = np.array([26,26], dtype=np.int32)

geom = CartesianGeometry(problem.xL,problem.xR, Ns, problem.geometry_folder, BC=problem.BC)

In [ ]:
FEM2D = Scipy2DFEM(geom,FEM1Dx, FEM1Dy, folder=problem.folderName)

In [ ]:
solver = DeCSpaceTimeSUPGSolver(problem, FEM2D, dec, GF=GF, stab=stab)

In [ ]:
problem.folderName

In [ ]:
filename = problem.folderName+"/final_sol_"+method_name+"_ord_%d_N_%04d.pkl"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0])
print(filename)
if False: #os.path.exists(filename):
    with open(filename, 'rb') as handle:
        final_sol = pickle.load(handle)
    q_final = final_sol[0]; 
    t = final_sol[1]; 
    comp_times=final_sol[2]; 
    err = final_sol[3]; 
    err_vertex = final_sol[4] 
else:

    q_save, tt_save, comp_times, err, err_vertex = solver.solve(with_error = True, with_error_vertex= True)

    q_final = dict()
    for var in problem.vars:
        q_final[var] = q_save[var][-1]
    sol_to_save = [q_final, tt_save[-1], comp_times, err, err_vertex ]
    # Open a file and use dump()
    savefile_name = filename
    with open(savefile_name, 'wb') as file:
        # A new file will be created
        pickle.dump(sol_to_save, file)

In [ ]:
err

In [ ]:

U = FEM2D.vect_to_mat(solver.ic_vect["u"])
V = FEM2D.vect_to_mat(solver.ic_vect["v"])
X = FEM2D.xx_mat[0]
Y = FEM2D.xx_mat[1]

levels = np.linspace(0,0.1,20)

Unorm= np.sqrt(U**2+V**2)
plt.figure(figsize=(4,3))
plt.contourf(X,Y,Unorm,levels=levels)
plt.axis("equal")
plt.colorbar()
plt.tight_layout()
plt.savefig(problem.folderName+"/final_sol_unorm_exact_ord_%d_N_%04d.pdf"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))

print(np.min(Unorm),np.max(Unorm))

In [ ]:

P = FEM2D.vect_to_mat(solver.ic_vect["p"])
X = FEM2D.xx_mat[0]
Y = FEM2D.xx_mat[1]

# levels = np.linspace(0,0.88,20)

Unorm= np.sqrt(U**2+V**2)
plt.figure(figsize=(4,3))
plt.contourf(X,Y,P,levels=20)
plt.axis("equal")
plt.colorbar()
plt.tight_layout()
plt.savefig(problem.folderName+"/final_sol_p_exact_ord_%d_N_%04d.pdf"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))

print(np.min(Unorm),np.max(Unorm))

In [ ]:
U = FEM2D.vect_to_mat(q_final["u"])
V = FEM2D.vect_to_mat(q_final["v"])
X = FEM2D.xx_mat[0]
Y = FEM2D.xx_mat[1]

levels = np.linspace(0,0.1,20)

Unorm= np.sqrt(U**2+V**2)
plt.figure(figsize=(4,3))
plt.contourf(X,Y,Unorm,levels=levels)
plt.axis("equal")
plt.colorbar()
plt.tight_layout()
plt.savefig(problem.folderName+"/final_sol_unorm_"+method_name+"_ord_%d_N_%04d.pdf"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))


print(np.min(Unorm),np.max(Unorm))

In [ ]:
P = FEM2D.vect_to_mat(q_final["p"])
X = FEM2D.xx_mat[0]
Y = FEM2D.xx_mat[1]


Unorm= np.sqrt(U**2+V**2)
plt.figure(figsize=(4,3))
plt.contourf(X,Y,P,levels=20)
plt.axis("equal")
plt.colorbar()
plt.tight_layout()
plt.savefig(problem.folderName+"/final_sol_p_"+method_name+"_ord_%d_N_%04d.pdf"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))


print(np.min(Unorm),np.max(Unorm))

In [ ]:
DIV = FEM2D.compute_discrete_divergence(q_final)

DIV = FEM2D.vect_to_mat(DIV)
X = FEM2D.xx_mat[0]
Y = FEM2D.xx_mat[1]

levels = 100#np.linspace(0,1.,20)

plt.figure(figsize=(6,5))
plt.contourf(X,Y,DIV,levels=levels)
plt.axis("equal")
plt.title("Divergence of solution "+method_name+" ord %d N %04d"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))
plt.colorbar()
plt.tight_layout()
plt.savefig(problem.folderName+"/final_div_"+method_name+"_ord_%d_N_%04d.pdf"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))


In [ ]:
# Z = np.sqrt(FEM2D.vect_to_mat(q_final["u"])**2+FEM2D.vect_to_mat(q_final["v"])**2)
Z = FEM2D.vect_to_mat(q_final["p"])
X = FEM2D.xx_mat[0]
Y = FEM2D.xx_mat[1]

%matplotlib widget
 
# importing required libraries
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot the surface.
surf = ax.plot_surface(X, Y, Z, cmap=cm.winter, #cm.cool,#cm.gnuplot2, #cm.coolwarm
                       linewidth=0, antialiased=False)

# Customize the z axis.
# ax.set_zlim(-1.01, 1.01)
ax.zaxis.set_major_locator(LinearLocator(7))
# A StrMethodFormatter is used automatically
# ax.zaxis.set_major_formatter('{x:1.2e}')

# Add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.7)#, aspect=5)

plt.show()